# Baseline Comparison

Runs all baseline models on the same evaluation sets and produces a unified comparison table.

| Model | Type | Notes |
|---|---|---|
| Presidio | Rule-based | Microsoft open-source PHI anonymizer |
| GLiNER | Zero-shot NER | `urchade/gliner_mediumv2.1` |
| RoBERTa-i2b2 | Fine-tuned NER | `obi/deid_roberta_i2b2` |
| scispaCy | Medical NER | `en_core_sci_lg` (not de-id specific) |
| EdgeDeID-SFT | Ours | Qwen3-0.6B SFT |
| EdgeDeID-GRPO | Ours | Qwen3-0.6B SFT + GRPO |

## 0 — Setup

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
from tqdm import tqdm

from pipeline.data.preprocessing import preprocess_text
from pipeline.data.xml_utils import convert_to_xml
from pipeline.eval import compute_metrics, print_report, to_dataframe

# ── dataset selection ────────────────────────────────────────────────────────
# "synth" = held-out synthetic test set
# "ext"   = external English benchmark
DATASET = "synth"   # change to "ext" for external benchmark

if DATASET == "synth":
    raw_df = pd.read_excel("synth_eval/LLM Eval.xlsx")
    raw_df.rename(columns={"Revision": "default_response_1"}, inplace=True)
    raw_df = preprocess_text(raw_df, input_col="default_response_0", output_col="default_response_1")
    raw_df["GT-XML"] = raw_df["default_response_1"].apply(convert_to_xml)
    NOTE_COL = "default_response_0"
else:
    raw_df = pd.read_csv("ext_eval/en_data.csv")
    raw_df = preprocess_text(raw_df, input_col="Report", output_col="DeID")
    raw_df["GT-XML"] = raw_df["DeID"].apply(convert_to_xml)
    NOTE_COL = "Report"

for tag in ["<anonymized_report>", "</anonymized_report>", "<anonymized_report/>",
            "<raw_report>", "</raw_report>", "<raw_report/>"]:
    raw_df[NOTE_COL] = raw_df[NOTE_COL].str.replace(tag, "", regex=False).str.strip()

notes     = raw_df[NOTE_COL].tolist()
gt_xmls   = raw_df["GT-XML"].tolist()
print(f"{len(notes)} evaluation samples | dataset={DATASET}")

## 1 — Load Baseline Models

Comment out any baselines whose dependencies are not installed.

In [ ]:
from baselines import PresidioBaseline, GLiNERBaseline, RobertaDeidBaseline, ScispaCyBaseline

models = [
    PresidioBaseline(score_threshold=0.5),
    GLiNERBaseline(model_name="urchade/gliner_mediumv2.1", threshold=0.5),
    RobertaDeidBaseline(model_name="obi/deid_roberta_i2b2"),
    ScispaCyBaseline(model_name="en_core_sci_lg"),
]
print("Loaded:", [m.name for m in models])

## 2 — Run Baselines

In [ ]:
all_results = {}   # model_name -> EvalResult

for model in models:
    print(f"\nRunning {model.name} ...")
    predictions = [
        model.predict_xml(note)
        for note in tqdm(notes, desc=model.name)
    ]
    result = compute_metrics(predictions, gt_xmls, source_notes=notes)
    all_results[model.name] = result
    print_report(result, title=model.name)

## 3 — Add Our Models

Load pre-computed predictions from the synth_eval / ext_eval notebooks
(run those notebooks first and save `*_predictions.csv`).

In [ ]:
import os

prefix = "synth_eval" if DATASET == "synth" else "ext_eval"

# SFT model predictions
sft_pred_path = f"{prefix}/{prefix.replace('_eval','')}_eval_predictions.csv"
if os.path.exists(sft_pred_path):
    sft_df = pd.read_csv(sft_pred_path)
    sft_result = compute_metrics(
        sft_df["LLM-XML"].tolist(), gt_xmls, source_notes=notes
    )
    all_results["EdgeDeID-SFT"] = sft_result
    print_report(sft_result, title="EdgeDeID-SFT")
else:
    print(f"SFT predictions not found at {sft_pred_path} — run synth_eval.ipynb first.")

# GRPO model predictions — update path after GRPO training
# grpo_pred_path = f"{prefix}/grpo_predictions.csv"
# if os.path.exists(grpo_pred_path):
#     grpo_df = pd.read_csv(grpo_pred_path)
#     grpo_result = compute_metrics(grpo_df["LLM-XML"].tolist(), gt_xmls, source_notes=notes)
#     all_results["EdgeDeID-GRPO"] = grpo_result
#     print_report(grpo_result, title="EdgeDeID-GRPO")

## 4 — Comparison Table

In [ ]:
rows = []
for model_name, r in all_results.items():
    rows.append({
        "Model":              model_name,
        "Micro-P":            round(r.micro_precision, 4),
        "Micro-R":            round(r.micro_recall, 4),
        "Micro-F1":           round(r.micro_f1, 4),
        "Macro-F1":           round(r.macro_f1, 4),
        "XML Parse Rate":     round(r.xml_parse_rate, 4),
        "Hallucination Rate": round(r.hallucination_rate, 4),
        "Token Reduction":    round(r.token_reduction_rate, 4) if r.token_reduction_rate == r.token_reduction_rate else None,
    })

comparison_df = pd.DataFrame(rows).set_index("Model")
comparison_df

In [ ]:
comparison_df.to_csv(f"baseline_comparison_{DATASET}.csv")
print(f"Saved baseline_comparison_{DATASET}.csv")

## 5 — Per-Entity Breakdown

In [ ]:
# F1 per entity type for each model — useful for the per-tag paper table
all_entity_tags = sorted(
    set(tag for r in all_results.values() for tag in r.per_entity)
)
per_entity_rows = []
for entity in all_entity_tags:
    row = {"Entity": entity}
    for model_name, r in all_results.items():
        m = r.per_entity.get(entity)
        row[model_name] = round(m.f1, 4) if m else None
    per_entity_rows.append(row)

per_entity_df = pd.DataFrame(per_entity_rows).set_index("Entity")
per_entity_df.to_csv(f"baseline_per_entity_{DATASET}.csv")
per_entity_df